# Лабораторная работа 3. Кэш с WAL

In [1]:
import os
from wal_cache import WalCache

WAL = 'demo.wal'
if os.path.exists(WAL):
    os.remove(WAL)

## Базовые операции с записью в WAL

In [2]:
cache = WalCache(WAL)
cache.set('a', 1)
cache.set('b', 'hello')
cache.set('c', [1, 2, 3])
cache.delete('a')
cache.set('b', 'world')
print('a =', cache.get('a'))
print('b =', cache.get('b'))
print('c =', cache.get('c'))

print('--- содержимое WAL ---')
with open(WAL) as f:
    print(f.read())

a = None
b = world
c = [1, 2, 3]
--- содержимое WAL ---
{"op": "set", "k": "a", "v": 1}
{"op": "set", "k": "b", "v": "hello"}
{"op": "set", "k": "c", "v": [1, 2, 3]}
{"op": "del", "k": "a"}
{"op": "set", "k": "b", "v": "world"}



## Восстановление состояния из WAL
Закроем текущий кэш и создадим новый — он должен прочитать WAL и восстановить состояние.

In [3]:
cache.close()
cache2 = WalCache(WAL)
print('a =', cache2.get('a'))
print('b =', cache2.get('b'))
print('c =', cache2.get('c'))
assert cache2.get('a') is None
assert cache2.get('b') == 'world'
assert cache2.get('c') == [1, 2, 3]
print('OK: состояние восстановлено')

a = None
b = world
c = [1, 2, 3]
OK: состояние восстановлено


## Компактизация WAL
После компактизации WAL содержит только актуальные ключи без истории операций.

In [4]:
print('размер WAL до компактизации:', os.path.getsize(WAL), 'байт')
with open(WAL) as f:
    print('строк:', sum(1 for _ in f))

cache2.compact()

print('размер WAL после компактизации:', os.path.getsize(WAL), 'байт')
with open(WAL) as f:
    lines = f.readlines()
print('строк:', len(lines))
for ln in lines:
    print(' ', ln.rstrip())

размер WAL до компактизации: 172 байт
строк: 5
размер WAL после компактизации: 78 байт
строк: 2
  {"op": "set", "k": "b", "v": "world"}
  {"op": "set", "k": "c", "v": [1, 2, 3]}


## Автоматическая компактизация (фоновый поток раз в минуту)
Для демонстрации создадим кэш с интервалом 1 секунда.

In [5]:
import time
WAL2 = 'auto.wal'
if os.path.exists(WAL2):
    os.remove(WAL2)
fast = WalCache(WAL2, compaction_interval=1.0)
for i in range(50):
    fast.set('x', i)
print('строк до автокомпактизации:', sum(1 for _ in open(WAL2)))
time.sleep(1.5)
print('строк после:', sum(1 for _ in open(WAL2)))
print('x =', fast.get('x'))
fast.close()
cache2.close()
for p in (WAL, WAL2):
    if os.path.exists(p):
        os.remove(p)

строк до автокомпактизации: 50
строк после: 1
x = 49
